# avgpool-reduce composite — cx18: AvgPool2d with stride: as_strided windowing + einops.reduce(mean)

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `avgpool-reduce`, `as-strided-windowing`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "avgpool-reduce"
DD_ATOM_IDS = ["avgpool-reduce", "as-strided-windowing"]
DD_SUBTOPICS = ["CNN: AvgPool as reduce", "PyTorch: as_strided windowing"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Same skeleton as cx17 (MaxPool), with `'mean'` swapped in for `'max'`. The windowing step is literally identical — the only difference is the reducer applied to the `(KH, KW)` axes after the view is built.

1. **`as-strided-windowing`** — build the `(B, C, OH, OW, KH, KW)` window view. OH/OW strides scale by pool stride `S`; KH/KW strides stay at the source spatial strides.
2. **`avgpool-reduce`** — `einops.reduce(..., '... kh kw -> ...', 'mean')` collapses the pool patch to its mean.

**Pop quiz: why does this compose so cleanly?** Pool is a *commutative-monoid reduction* over the within-window axes. Both `max` and `mean` are such reductions; so is `sum` and `prod`. Once you have the window view, ANY reducer plugs in — the view is the universal infrastructure.

**ARENA AdaptiveAvgPool, briefly.** Global avg-pool (used at the end of ResNet) is the degenerate case: `K = H, S = H`, OH = OW = 1. The composite handles it without modification.

### Composite Exercise — AvgPool2d with stride: as_strided windowing + einops.reduce(mean)

**Atoms exercised together**: `avgpool-reduce`, `as-strided-windowing`

Implement `cx18_avgpool2d(x, kernel_size, stride=None)`.

- `x`: float tensor `(B, C, H, W)`.
- `kernel_size`: int (square kernel).
- `stride`: int or None — default is `kernel_size` (non-overlapping).
- Return: tensor `(B, C, OH, OW)` matching `F.avg_pool2d(x, kernel_size, stride=stride)` (no padding, default `count_include_pad=True` doesn't matter since we don't pad).

Same recipe as cx17 — only the reducer changes. The test verifies overlapping and non-overlapping cases against `F.avg_pool2d`.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx18_avgpool2d(x, kernel_size, stride=None):
    raise NotImplementedError

def _test_cx18():
    from torch.nn import functional as F
    rng = t.Generator().manual_seed(18)

    # Case A: hand example.
    x = t.tensor([[[
        [1.0, 2.0, 3.0, 4.0],
        [5.0, 6.0, 7.0, 8.0],
        [9.0, 1.0, 2.0, 3.0],
        [4.0, 5.0, 6.0, 7.0],
    ]]])
    y = cx18_avgpool2d(x, kernel_size=2)
    expected = t.tensor([[[
        [(1+2+5+6)/4, (3+4+7+8)/4],
        [(9+1+4+5)/4, (2+3+6+7)/4],
    ]]])
    assert tuple(y.shape) == (1, 1, 2, 2)
    assert t.allclose(y, expected, atol=1e-6)

    # Case B: overlapping and non-overlapping cross-checks.
    for B,C,H,W,K,S in [(2,3,8,8,3,2),(1,1,16,16,3,2),(2,4,10,12,2,1),(1,2,8,8,4,4),(3,2,12,12,2,3)]:
        xr = t.randn(B, C, H, W, generator=rng)
        yours = cx18_avgpool2d(xr, kernel_size=K, stride=S)
        yref = F.avg_pool2d(xr, kernel_size=K, stride=S)
        assert tuple(yours.shape) == tuple(yref.shape), f'shape on {(B,C,H,W,K,S)}'
        assert t.allclose(yours, yref, atol=1e-5), f'value on {(B,C,H,W,K,S)}'

    # Case C: default stride.
    xr = t.randn(2, 3, 8, 8, generator=rng)
    yours = cx18_avgpool2d(xr, kernel_size=4)
    yref = F.avg_pool2d(xr, kernel_size=4)
    assert t.allclose(yours, yref, atol=1e-5)

    # Case D: constant input → constant output (mean of constants = constant).
    xc = t.full((2, 3, 6, 6), 4.2)
    yc = cx18_avgpool2d(xc, kernel_size=3, stride=3)
    assert tuple(yc.shape) == (2, 3, 2, 2)
    assert t.allclose(yc, t.full((2, 3, 2, 2), 4.2), atol=1e-6)

    # Case E: global pool — K==H, S==H → OH==OW==1.
    xg = t.randn(2, 4, 7, 7, generator=rng)
    yg = cx18_avgpool2d(xg, kernel_size=7, stride=7)
    assert tuple(yg.shape) == (2, 4, 1, 1)
    assert t.allclose(yg, F.avg_pool2d(xg, kernel_size=7, stride=7), atol=1e-5)
    _dd_passed.add('cx18')

_test_cx18()

<details><summary>Show solution — cx18</summary>

```python
def cx18_avgpool2d(x, kernel_size, stride=None):
    K = kernel_size
    S = K if stride is None else stride
    B, C, H, W = x.shape
    OH = (H - K) // S + 1
    OW = (W - K) // S + 1
    # Atom A (as-strided-windowing): pool window view, no copy.
    s_b, s_c, s_h, s_w = x.stride()
    x_win = x.as_strided(
        size=(B, C, OH, OW, K, K),
        stride=(s_b, s_c, s_h * S, s_w * S, s_h, s_w),
    )
    # Atom B (avgpool-reduce): collapse within-window axes by mean.
    return einops.reduce(x_win, 'b c oh ow kh kw -> b c oh ow', 'mean')
```

Compare against cx17: the only difference is `'max' -> 'mean'`. The lesson is that pool operations are NOT atomic — they decompose into windowing (a view-construction trick) + reduction (a one-letter parameter). Recognising this turns AvgPool / MaxPool / SumPool / global-pool into the same op with different reducers.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx18'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx18',
        'subtopics': ["CNN: AvgPool as reduce", "PyTorch: as_strided windowing"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()